# 05 Stratified Analysis and Confounding — Reference Solutions

Complete solutions to the stratified analysis exercises for the Songbai Nursing Home Legionnaires' disease cluster.

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## Question 1: Confounding Analysis of Hydrotherapy Use

In [ ]:
# --- Crude RR ---
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a0 = int(ct_hydro.loc[1, 1])
b0 = int(ct_hydro.loc[1, 0])
c0 = int(ct_hydro.loc[0, 1])
d0 = int(ct_hydro.loc[0, 0])
crude_rr_hydro = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"Crude RR (hydrotherapy -> infected) = {crude_rr_hydro:.3f}")

# --- Verify the confounder conditions ---
print("\n=== Functional status x Hydrotherapy use rate ===")
print(pd.crosstab(df["functional_status"], df["hydrotherapy_use"],
                  normalize="index").round(3))

# --- Stratum-specific RR ---
strata = sorted(df["functional_status"].unique())
hydro_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["hydrotherapy_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {s}: skipped")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    hydro_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

hydro_df = pd.DataFrame(hydro_results)
print("\n=== Stratum-specific RR ===")
for _, row in hydro_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  Crude RR = {crude_rr_hydro:.3f}")

## Question 2: Mantel-Haenszel Adjustment

In [ ]:
numerator = 0
denominator = 0

for _, row in hydro_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_hydro = numerator / denominator

print(f"Mantel-Haenszel adjusted RR = {rr_mh_hydro:.3f}")
print(f"Crude RR                    = {crude_rr_hydro:.3f}")
print(f"Difference                  = {crude_rr_hydro - rr_mh_hydro:.3f}")

if abs(crude_rr_hydro - rr_mh_hydro) > 0.1:
    print("\n-> Functional status is indeed a confounder of hydrotherapy use (the crude RR was inflated)")
else:
    print("\n-> After controlling for functional status the RR barely changed; confounding effect is limited")

## Question 3 (Challenge): Stratify by Age Group + Forest Plot

In [ ]:
# Crude RR
ct_shower = pd.crosstab(df["shower_use"], df["infected"])
a_crude = int(ct_shower.loc[1, 1])
b_crude = int(ct_shower.loc[1, 0])
c_crude = int(ct_shower.loc[0, 1])
d_crude = int(ct_shower.loc[0, 0])
crude_rr = risk_ratio(a_crude, a_crude + b_crude, c_crude, c_crude + d_crude)

# Create age groups
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# Stratum-specific RR
age_results = []
for grp in ["60-69", "70-79", "80-89", "90+"]:
    sub = df[df["age_group"] == grp]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {grp}: skipped")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    age_results.append({
        "stratum": grp, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

age_df = pd.DataFrame(age_results)
print("=== Stratum-specific RR by age group ===")
for _, row in age_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")

In [ ]:
# Forest plot
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(age_df))

ax.errorbar(
    age_df["RR"], y_pos,
    xerr=[age_df["RR"] - age_df["CI_lower"],
          age_df["CI_upper"] - age_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"Crude RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(age_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("Forest plot: shower use -> infection (stratified by age group)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MH adjusted RR
num = 0
den = 0
for _, row in age_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    num += a_i * (c_i + d_i) / n_i
    den += c_i * (a_i + b_i) / n_i

rr_mh_age = num / den

print(f"MH adjusted RR (controlling for age) = {rr_mh_age:.3f}")
print(f"Crude RR                             = {crude_rr:.3f}")
print(f"Difference                           = {crude_rr - rr_mh_age:.3f}")

# Homogeneity
rr_vals = age_df["RR"].values
print(f"\nRange of stratum-specific RRs: {rr_vals.min():.3f} – {rr_vals.max():.3f}")
if rr_vals.max() - rr_vals.min() > 0.5:
    print("-> The RRs across age groups differ substantially; age effect modification may be present")
else:
    print("-> The RRs across age groups are similar; age interaction is not notable")

### Interpretation

- **Functional status**: bedridden residents don't shower and are also less often infected; ambulatory residents shower more and are also more often infected → classic confounding
- **After MH adjustment**: if RR_MH is clearly smaller than the crude RR, functional status is confirmed as a confounder
- **Age stratification**: if the RRs across age groups are close, age interaction is small
- **Limitation**: stratified analysis can control for only one variable at a time → we need Ch06's logistic regression to adjust for multiple factors simultaneously